In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/experiment_for_xi/validate_cka_online/resnet50_from_scratch_on_cifar100_2

/content/drive/MyDrive/experiment_for_xi/validate_cka_online/resnet50_from_scratch_on_cifar100_2


In [1]:
import os
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
import pickle
from matplotlib import pyplot as plt
from mpl_toolkits import axes_grid1
import numpy as np
import random
import tqdm
from torch.utils.data import Subset
from functools import partial
from typing import List, Dict
import gc
from torch.optim.lr_scheduler import _LRScheduler
import math
from torchvision.transforms.autoaugment import AutoAugment, RandAugment, AutoAugmentPolicy
from torchvision.transforms import v2

In [2]:
class Arguments:
 def __init__(self, epochs=160, batch_size=128, seed=42, warm=10, save_fully_trained_reference_model=True, compute_CKA_online=False, fully_trained_reference_model="best_model.pt", re_size=1024, variance_threshold=0.0002, moving_window=20, cka_value_cutoff=0.3, stride=5):
  self.epochs = epochs
  self.batch_size = batch_size
  self.seed = seed
  self.warm = warm
  self.save_fully_trained_reference_model = save_fully_trained_reference_model
  self.compute_CKA_online = compute_CKA_online
  self.fully_trained_reference_model = fully_trained_reference_model
  self.re_size = re_size
  self.variance_threshold = variance_threshold
  self.moving_window = moving_window
  self.cka_value_cutoff = cka_value_cutoff
  self.stride = stride

In [3]:
def seed_worker(worker_id):
  """
  Ensure reproducibility for each worker in DataLoader.
  """
  worker_seed = torch.initial_seed() % 2 ** 32
  np.random.seed(worker_seed)
  random.seed(worker_seed)

In [4]:
def add_colorbar(im, aspect=10, pad_fraction=0.5, **kwargs):
    """Add a vertical color bar to an image plot."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1./aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes("right", size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)

In [5]:
def random_sample(input_tensor, size):
  rand_num = set()
  input_tensor = input_tensor.reshape(-1)
  while 1:
    rand_num.add(random.randint(0, len(input_tensor)-1))
    if len(rand_num) >= size:
      break
  result = []
  rand_num = sorted(rand_num)
  for i in rand_num:
      result.append(input_tensor[i].item())
  result = torch.tensor(result).unsqueeze(0)
  return result

In [6]:
def is_target_layer(module):
  # Check if the module has no children
  has_no_children = len(list(module.children())) == 0

  # Check if the module is a Conv2d
  is_target_type = isinstance(module, (nn.Conv2d))

  return has_no_children and is_target_type

In [7]:
def is_bn_layer(module):
  # Check if the module has no children
  has_no_children = len(list(module.children())) == 0

  # Check if the module is a BatchNorm2d layer
  is_target_type = isinstance(module, (nn.BatchNorm2d))

  return has_no_children and is_target_type

In [8]:
class WarmUpLR(_LRScheduler):
    """warmup_training learning rate scheduler
    Args:
        optimizer: optimzier(e.g. SGD)
        total_iters: totoal_iters of warmup phase
    """
    def __init__(self, optimizer, total_iters, last_epoch=-1):

        self.total_iters = total_iters
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        """we will use the first m batches, and set the learning
        rate to base_lr * m / total_iters
        """
        return [base_lr * self.last_epoch / (self.total_iters + 1e-8) for base_lr in self.base_lrs]

In [9]:
def soft_cross_entropy(outputs, soft_targets):
    log_probs = torch.nn.functional.log_softmax(outputs, dim=1)
    return -torch.mean(torch.sum(soft_targets * log_probs, dim=1))

In [10]:
class CKA:
    def __init__(self,
                 model1: nn.Module,
                 model2: nn.Module,
                 model1_name: str = None,
                 model2_name: str = None,
                 model1_layers: List[str] = None,
                 model2_layers: List[str] = None,
                 device: str = 'cuda'):
        """
        :param model1: (nn.Module) model 1
        :param model2: (nn.Module) model 2
        :param model1_name: (str) name of model 1
        :param model2_name: (str) name of model 2
        :param model1_layers: (List[str]) list of layers to compare for model 1
        :param model2_layers: (List[str]) list of layers to compare for model 2
        :param device: (str) device to run the model on
        """

        self.model1 = model1
        self.model2 = model2

        self.device = device

        self.model1_info = {}
        self.model2_info = {}

        if model1_name is None:
            self.model1_info['Name'] = model1.__repr__().split('(')[0]
        else:
            self.model1_info['Name'] = model1_name

        if model2_name is None:
            self.model2_info['Name'] = model2.__repr__().split('(')[0]
        else:
            self.model2_info['Name'] = model2_name

        self.model1_info['Layers'] = []
        self.model2_info['Layers'] = []

        self.model1_features = {}
        self.model2_features = {}

        self.model1_layers = model1_layers
        self.model2_layers = model2_layers

        self._insert_hooks()

        self.model1.eval()
        self.model2.eval()

    def _log_layer(self,
                   model: str,
                   name: str,
                   layer: nn.Module,
                   inp: torch.Tensor,
                   out: torch.Tensor):
        # Ensure activations are explicity on the CPU
        out = out.detach().cpu()
        if model == "model1":
            self.model1_features[name] = out
        elif model == "model2":
            self.model2_features[name] = out
        else:
            raise RuntimeError("Unknown model name for _log_layer")

    def _insert_hooks(self):
        self.hook_handles = []
        # Model 1
        for name, layer in self.model1.named_modules():
            if self.model1_layers is not None:
                if name in self.model1_layers:
                    self.model1_info['Layers'] += [name]
                    handle = layer.register_forward_hook(partial(self._log_layer, "model1", name))
                    self.hook_handles.append(handle)
            else:
                self.model1_info['Layers'] += [name]
                handle = layer.register_forward_hook(partial(self._log_layer, "model1", name))
                self.hook_handles.append(handle)
        # Model 2
        for name, layer in self.model2.named_modules():
            if self.model2_layers is not None:
                if name in self.model2_layers:
                    self.model2_info['Layers'] += [name]
                    handle = layer.register_forward_hook(partial(self._log_layer, "model2", name))
                    self.hook_handles.append(handle)
            else:
                self.model2_info['Layers'] += [name]
                handle = layer.register_forward_hook(partial(self._log_layer, "model2", name))
                self.hook_handles.append(handle)

    def _HSIC(self, K, L):
        """
        Computes the unbiased estimate of HSIC metric.

        Reference: https://arxiv.org/pdf/2010.15327.pdf Eq (3)
        """
        N = K.shape[0]
        ones = torch.ones(N, 1)
        result = torch.trace(K @ L)
        result += ((ones.t() @ K @ ones @ ones.t() @ L @ ones) / ((N - 1) * (N - 2))).item()
        result -= ((ones.t() @ K @ L @ ones) * 2 / (N - 2)).item()
        return (1 / (N * (N - 3)) * result).item()
    def compare(self,
                dataloader1: DataLoader,
                dataloader2: DataLoader = None,
                num_times_iterate_over_test_dataset=1) -> None:
        """
        Computes the feature similarity between the models on the
        given datasets.
        :param dataloader1: (DataLoader)
        :param dataloader2: (DataLoader) If given, model 2 will run on this
                            dataset. (default = None)
        """
        if dataloader2 is None:
            dataloader2 = dataloader1

        self.model1_info['Dataset'] = dataloader1.dataset.__repr__().split('\n')[0]
        self.model2_info['Dataset'] = dataloader2.dataset.__repr__().split('\n')[0]

        N = len(self.model1_layers) if self.model1_layers is not None else len(list(self.model1.modules()))
        M = len(self.model2_layers) if self.model2_layers is not None else len(list(self.model2.modules()))

        self.hsic_matrix = torch.zeros(N, M, 3, device="cpu")

        # num_batches = min(len(dataloader1), len(dataloader2)) * num_times_iterate_over_test_dataset
        num_batches = 10  # only want to go through 10 batches this time
        # print(f"Total number of batches in use: {num_batches}")

        smaller_total = 10 # Reduce number of batches processed, boring to wait

        for _ in range(num_times_iterate_over_test_dataset):
          for (x1, *_) in tqdm.tqdm(dataloader1, desc="| Comparing features |", total=smaller_total):
              if smaller_total == 0:
                break
              smaller_total -= 1
              self.model1_features = {}
              self.model2_features = {}
              with torch.no_grad():
                # Forward pass remains on GPU
                _ = self.model1(x1.to(self.device))
                _ = self.model2(x1.to(self.device))

                for i, (name1, feat1) in enumerate(self.model1_features.items()):
                    X = feat1.flatten(1).cpu() # Move features to CPU
                    # print(f"Shape of Activation X from layer {name1}: {X.shape}")
                    K = X @ X.t()
                    K.fill_diagonal_(0.0)
                    self.hsic_matrix[i, :, 0] += self._HSIC(K, K)

                    for j, (name2, feat2) in enumerate(self.model2_features.items()):
                        Y = feat2.flatten(1).cpu() # Move features to CPU
                        # print(f"Shape of Activation Y from layer {name2}: {Y.shape}")
                        L = Y @ Y.t()
                        L.fill_diagonal_(0.0)
                        assert K.shape == L.shape, f"Feature shape mistach! {K.shape}, {L.shape}"

                        self.hsic_matrix[i, j, 1] += self._HSIC(K, L)
                        self.hsic_matrix[i, j, 2] += self._HSIC(L, L)
              # Explicitly delete all intermediate tensors
              del X, K, Y, L
              torch.cuda.empty_cache()
              gc.collect()
        self.hsic_matrix[:, :, 0] /= num_batches
        self.hsic_matrix[:, :, 1] /= num_batches
        self.hsic_matrix[:, :, 2] /= num_batches
        self.hsic_matrix = self.hsic_matrix[:, :, 1] / (self.hsic_matrix[:, :, 0].sqrt() *
                                                        self.hsic_matrix[:, :, 2].sqrt())

        assert not torch.isnan(self.hsic_matrix).any(), "HSIC computation resulted in NANs"

    def export(self) -> Dict:
        """
        Exports the CKA data along with the respective model layer names.
        :return:
        """
        return {
            "model1_name": self.model1_info['Name'],
            "model2_name": self.model2_info['Name'],
            "CKA": self.hsic_matrix,
            "model1_layers": self.model1_info['Layers'],
            "model2_layers": self.model2_info['Layers'],
            "dataset1_name": self.model1_info['Dataset'],
            "dataset2_name": self.model2_info['Dataset']
        }

    def plot_results(self,
                     save_path: str = None,
                     title: str = None):
        fig, ax = plt.subplots()
        im = ax.imshow(self.hsic_matrix, origin='lower', cmap='magma')
        ax.set_xlabel(f"Layers {self.model2_info['Name']}", fontsize=15)
        ax.set_ylabel(f"Layers {self.model1_info['Name']}", fontsize=15)

        if title is not None:
            ax.set_title(f"{title}", fontsize=18)
        else:
            ax.set_title(f"{self.model1_info['Name']} vs {self.model2_info['Name']}", fontsize=18)

        add_colorbar(im)
        plt.tight_layout()

        if save_path is not None:
            plt.savefig(save_path, dpi=300)

        plt.show()

    def __del__(self):
      # print("Deleting CKA object and freeing memory...")

      # Removing handles
      for handle in self.hook_handles:
        handle.remove()
      self.hook_handles.clear()

      # Remove models safely
      if hasattr(self, "model1"):
        del self.model1
      if hasattr(self, "model2"):
        del self.model2

      # Remove feature storage
      self.model1_features.clear()
      self.model2_features.clear()

      # Remove layer information
      self.model1_layers = None
      self.model2_layers = None
      self.model1_info.clear()
      self.model2_info.clear()

      # Remove HSIC matrix
      if hasattr(self, "hsic_matrix"):
          del self.hsic_matrix

      # Run garbage collection and clear CUDA memory
      gc.collect()
      torch.cuda.empty_cache()

In [11]:
def main(args):
  best_acc = 0.0

  torch.manual_seed(args.seed)
  torch.cuda.manual_seed(args.seed)
  torch.cuda.manual_seed_all(args.seed)
  random.seed(args.seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

  # Create generator for DataLoader
  g = torch.Generator()
  g.manual_seed(args.seed)

  transforms_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    AutoAugment(AutoAugmentPolicy.CIFAR10),  # Apply AutoAugment
    RandAugment(),  # Randomly chosen augmentations
    transforms.ToTensor(),
    transforms.Normalize((0.5070758, 0.4865503, 0.44091913), (0.26733428, 0.25643846, 0.27615047)),
  ])

  transforms_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5070758, 0.4865503, 0.44091913), (0.26733428, 0.25643846, 0.27615047)),
  ])

  trainset = torchvision.datasets.CIFAR100(root='data', train=True, download=True, transform=transforms_train)
  testset = torchvision.datasets.CIFAR100(root='data', train=False, download=True, transform=transforms_test)

  num_classes = 100
  model = resnet50(weights=None)
  # Modify the first convolution layer to accept 3x32x32 inputs
  model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
  # Adjust the maxpool layer (since CIFAR-100 is small)
  model.maxpool = nn.Identity()  # Remove max pooling
  # Modify the last fully connected layer for CIFAR-100
  num_features = model.fc.in_features
  model.fc = nn.Linear(num_features, num_classes)

  net = model
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  net.to(device)

  if args.compute_CKA_online:
    fully_trained_model = resnet50(weights=None)
    fully_trained_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    fully_trained_model.maxpool = nn.Identity()
    num_features = fully_trained_model.fc.in_features
    fully_trained_model.fc = nn.Linear(num_features, num_classes)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.to(device)
    fully_trained_model = fully_trained_model.to(device)
    torch_state = torch.load(args.fully_trained_reference_model)
    fully_trained_model.load_state_dict(torch_state)
    fully_trained_reference_model = fully_trained_model

  criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

  num_workers = min(8, os.cpu_count() // 2)  # Use half of available cores
  trainloader = torch.utils.data.DataLoader(trainset, batch_size=args.batch_size, shuffle=True, num_workers=num_workers, worker_init_fn=seed_worker, generator=g)
  testloader = torch.utils.data.DataLoader(testset, batch_size=args.batch_size, shuffle=False, num_workers=num_workers, worker_init_fn=seed_worker, generator=g)
  cka_loader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=True, num_workers=num_workers, drop_last=True)

  optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
  train_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=args.epochs - args.warm, eta_min=1e-3) # learning rate decay
  iter_per_epoch = len(trainloader)
  warmup_scheduler = WarmUpLR(optimizer, iter_per_epoch * args.warm)


  if args.compute_CKA_online:
    # Once a layer is frozen, we will no longer need to compute CKA for it
    tracker_of_cka_values_across_epochs = {}
    tracker_of_cka_window_values_across_epochs = {}
    # Keep track of which layers have and have not been frozen to avoid computation
    cka_freeze_layer_configuration = {}
    for name, module in net.named_modules():
      if is_target_layer(module):
        tracker_of_cka_values_across_epochs[name] = []
        tracker_of_cka_window_values_across_epochs[name] = []
        cka_freeze_layer_configuration[name] = -1

  # Include additional information necessary to collect
  accuracy_list = []
  training_loss_list = []
  testing_loss_list = []

  # Perform Data Augmentation for Soft Labels
  cutmix = v2.CutMix(num_classes=num_classes)
  mixup = v2.MixUp(num_classes=num_classes)
  cutmix_or_mixup = transforms.RandomChoice([cutmix, mixup])

  for epoch in range(args.epochs):
    if epoch >= args.warm:
      train_scheduler.step(epoch)

    net.train()
    train_running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
      inputs, labels = data[0].to(device), data[1].to(device)
      inputs, labels = cutmix_or_mixup(inputs, labels)

      optimizer.zero_grad()

      outputs = net(inputs)
      loss = soft_cross_entropy(outputs, labels)
      loss.backward()
      optimizer.step()

      train_running_loss += loss.item()

      if epoch < args.warm:
        warmup_scheduler.step()

    train_running_loss /= len(trainloader)

    if args.compute_CKA_online:
      # CKA computation between the current network and the fully trained reference model
      # Update appropriate dictionaries and check for freezing
      model_layers = []
      for name, module in net.named_modules():
        if is_target_layer(module):
          model_layers.append(name)

      is_previous_layer_frozen = True

      for model_layer in model_layers:
        # Layer has not been frozen yet, need to compute CKA
        if cka_freeze_layer_configuration[model_layer] == -1:
          fully_trained_model = resnet50(weights=None)
          fully_trained_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
          fully_trained_model.maxpool = nn.Identity()
          num_features = fully_trained_model.fc.in_features
          fully_trained_model.fc = nn.Linear(num_features, num_classes)
          device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
          fully_trained_model = fully_trained_model.to(device)
          torch_state = torch.load(args.fully_trained_reference_model)
          fully_trained_model.load_state_dict(torch_state)
          fully_trained_reference_model = fully_trained_model

          print(f"Comparing for {model_layer} at epoch {epoch}")
          cka = CKA(net, fully_trained_reference_model, model1_name=f'ResNet50_{epoch}_Epoch', model2_name='ResNet50_Fully_Trained', model1_layers=[model_layer], model2_layers=[model_layer], device=device)
          with torch.no_grad():
            cka.compare(cka_loader, None, num_times_iterate_over_test_dataset=1)
          output_cka_dict = cka.export()
          tracker_of_cka_values_across_epochs[model_layer].append(output_cka_dict['CKA'][0].item())
          tracker_of_cka_window_values_across_epochs[model_layer].append(output_cka_dict['CKA'][0].item())

          if len(tracker_of_cka_window_values_across_epochs[model_layer]) > args.moving_window:
            # Remove previous CKA value if we have over the window size
            tracker_of_cka_window_values_across_epochs[model_layer] = tracker_of_cka_window_values_across_epochs[model_layer][args.stride:]

          if len(tracker_of_cka_window_values_across_epochs[model_layer]) == args.moving_window:
            # Begin checking variance threshold if we reached that point, use sample variance for unbiased estimate
            window_variance = np.var(tracker_of_cka_window_values_across_epochs[model_layer], ddof=1)

            current_cka_value = output_cka_dict['CKA'][0].item()
            if (window_variance < args.variance_threshold) and current_cka_value > args.cka_value_cutoff and is_previous_layer_frozen:
              cka_freeze_layer_configuration[model_layer] = epoch
              print(f"Froze layer {model_layer} at epoch {epoch}")

              # Signal to help freeze the corresponding batch normalization layer
              froze_cnn = False
              for name, module in net.named_modules():
                if is_target_layer(module) and name == model_layer:
                  froze_cnn = True

                  module.weight.requires_grad = False
                  if hasattr(module, 'bias') and module.bias is not None:
                    module.bias.requires_grad = False
                elif is_bn_layer(module) and froze_cnn:
                  module.weight.requires_grad = False
                  if hasattr(module, 'bias') and module.bias is not None:
                    module.bias.requires_grad = False
                  # Break out since we have already frozen corresponding batch normalization layer
                  break
            else:
              is_previous_layer_frozen = False
          # Explicitly call __del__ before deleting
          cka.__del__()
          del cka
          del output_cka_dict
          gc.collect() # Force garbage collection
          torch.cuda.empty_cache() # Free cached memory

    net.eval()
    totals = 0
    correct = 0
    testing_loss = 0
    with torch.no_grad():
      for data in testloader:
        inputs, labels = data[0].to(device), data[1].to(device)
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        totals += labels.size(0)
        correct += (predicted == labels).sum().item()
        testing_loss += criterion(outputs, labels).item()

    testing_loss /= len(testloader)
    accuracy = correct / totals
    print("Accuracy: ", accuracy, "Training Loss: ", train_running_loss, "Testing Loss: ", testing_loss)

    # Save metrics
    testing_loss_list.append(testing_loss)
    accuracy_list.append(accuracy)
    training_loss_list.append(train_running_loss)

    if args.save_fully_trained_reference_model:
      if accuracy > best_acc:
        checkpoint = net.state_dict()
        torch.save(checkpoint, "best_model.pt")
        print("New best accuracy:", accuracy)
        best_acc = accuracy

  # Save metrics depending on what mode you are using
  if args.save_fully_trained_reference_model:
    with open("fully_trained_reference_model_resnet50_metrics.pkl", "wb") as f:
      pickle.dump((accuracy_list, training_loss_list, testing_loss_list), f)
  elif args.compute_CKA_online:
    with open("compute_CKA_online_resnet50_metrics.pkl", "wb") as f:
      pickle.dump((accuracy_list, training_loss_list, testing_loss_list), f)

  if args.compute_CKA_online:
    with open("resnet50_cka_values_across_epochs.pkl", "wb") as f:
      pickle.dump(tracker_of_cka_values_across_epochs, f)
    with open("resnet50_cka_window_values_across_epochs.pkl", "wb") as f:
      pickle.dump(tracker_of_cka_window_values_across_epochs, f)
    with open("resnet50_cka_freeze_configuration.pkl", "wb") as f:
      pickle.dump(cka_freeze_layer_configuration, f)

In [12]:
args = Arguments()
main(args)

Files already downloaded and verified
Files already downloaded and verified
Accuracy:  0.0283 Training Loss:  4.752281164574196 Testing Loss:  5.044127349612079
New best accuracy: 0.0283
Accuracy:  0.054 Training Loss:  4.560413799627358 Testing Loss:  4.437540319901478
New best accuracy: 0.054
Accuracy:  0.0779 Training Loss:  4.448252418157085 Testing Loss:  4.15499698361264
New best accuracy: 0.0779
Accuracy:  0.0951 Training Loss:  4.38253180267256 Testing Loss:  4.032069761541825
New best accuracy: 0.0951
Accuracy:  0.11 Training Loss:  4.313528259696863 Testing Loss:  3.9496420425704764
New best accuracy: 0.11
Accuracy:  0.0987 Training Loss:  4.24242593321349 Testing Loss:  3.9498008534878113
Accuracy:  0.1388 Training Loss:  4.173249592866434 Testing Loss:  3.758136830752409
New best accuracy: 0.1388
Accuracy:  0.1874 Training Loss:  4.03939586893067 Testing Loss:  3.5006718333763414
New best accuracy: 0.1874
Accuracy:  0.2459 Training Loss:  3.910375470090705 Testing Loss:  3.

/home/idies/workspace/Storage/ktuzinows1/persistent/pytorch_env/PleaseWork/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Accuracy:  0.302 Training Loss:  3.7772635584292202 Testing Loss:  3.0626509038707876
New best accuracy: 0.302
Accuracy:  0.3082 Training Loss:  3.693059016371627 Testing Loss:  3.0913443746446054
New best accuracy: 0.3082
Accuracy:  0.283 Training Loss:  3.6419421010614967 Testing Loss:  3.1616443923757047
Accuracy:  0.3159 Training Loss:  3.5983978647100345 Testing Loss:  3.0576651911192303
New best accuracy: 0.3159
Accuracy:  0.3186 Training Loss:  3.579017830626739 Testing Loss:  3.050506643102139
New best accuracy: 0.3186
Accuracy:  0.3588 Training Loss:  3.5772618905967457 Testing Loss:  2.9212053365345243
New best accuracy: 0.3588
Accuracy:  0.4034 Training Loss:  3.5391143496384094 Testing Loss:  2.7309860730473
New best accuracy: 0.4034
Accuracy:  0.3725 Training Loss:  3.500880032244241 Testing Loss:  2.78674189350273
Accuracy:  0.3984 Training Loss:  3.482752848159322 Testing Loss:  2.7302314390110065
Accuracy:  0.4205 Training Loss:  3.4532493746189203 Testing Loss:  2.6403

In [ ]:
# Start computing similarity-guided training here
args = Arguments(save_fully_trained_reference_model=False, compute_CKA_online=True, moving_window=20, cka_value_cutoff=0.3, stride=5, variance_threshold=0.0002)
main(args)

Files already downloaded and verified
Files already downloaded and verified


/tmp/ipykernel_3935635/788301582.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch_state = torch.load(args.fully_trained_reference_model)
/tmp/ipykernel_3935635/788

Comparing for conv1 at epoch 0


| Comparing features |: 100%|██████████| 10/10 [00:03<00:00,  2.78it/s]


Comparing for layer1.0.conv1 at epoch 0


...

| Comparing features |: 100%|██████████| 10/10 [00:04<00:00,  2.34it/s]


Comparing for layer2.1.conv1 at epoch 99


| Comparing features |: 100%|██████████| 10/10 [00:03<00:00,  3.10it/s]


Comparing for layer2.1.conv2 at epoch 99


| Comparing features |:  70%|███████   | 7/10 [00:02<00:00,  3.50it/s]